<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/Cohort_2_Lec_5_HandsOn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install faiss-cpu sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [2]:
import os, re, time, json, textwrap
import numpy as np
import faiss

np.random.seed(0)

def show(title, rows):
    """Tiny pretty-printer so every result in this lab looks the same."""
    print("=" * 78)
    print(title)
    print("=" * 78)
    for r in rows:
        print(r)
    print()

def show_hits(title, hits):
    """hits = list of dicts with keys: score, id, dept, source, text"""
    print("=" * 78)
    print(title)
    print("=" * 78)
    for rank, h in enumerate(hits, 1):
        head = f"{rank}. [{h['score']:+.3f}] ({h['dept']:<4} | {h['source']})"
        body = textwrap.fill(h["text"], width=74, initial_indent="     ",
                             subsequent_indent="     ")
        print(head)
        print(body)
    print()

print("faiss version:", faiss.__version__)

faiss version: 1.15.0


In [3]:
CORPUS = [
    # ---------------- HR ----------------
    dict(id="hr-01", dept="HR", source="hr_policy.pdf", year=2024,
         text="Full-time employees accrue 24 days of paid annual leave per calendar year. Unused leave up to 10 days may be carried into the next year."),
    dict(id="hr-02", dept="HR", source="hr_policy.pdf", year=2024,
         text="Employees are entitled to 12 paid sick days per year. A medical certificate is required for any absence longer than three consecutive days."),
    dict(id="hr-03", dept="HR", source="hr_policy.pdf", year=2024,
         text="The standard parental leave policy grants 26 weeks of paid leave to the primary caregiver and 4 weeks to the secondary caregiver."),
    dict(id="hr-04", dept="HR", source="hr_policy.pdf", year=2024,
         text="Contract and temporary staff are not covered by the standard parental leave policy. Their entitlements are defined individually in the vendor agreement."),
    dict(id="hr-05", dept="HR", source="benefits_handbook.pdf", year=2024,
         text="The employee car-leasing benefit allows staff at grade M3 and above to lease a vehicle through payroll deduction over a 36-month term."),
    dict(id="hr-06", dept="HR", source="benefits_handbook.pdf", year=2024,
         text="Health insurance covers the employee, spouse and two dependent children up to a sum insured of eight lakh rupees per year."),
    dict(id="hr-07", dept="HR", source="benefits_handbook.pdf", year=2023,
         text="In the 2023 handbook the health insurance sum insured was five lakh rupees. This figure was revised upward in the 2024 revision."),
    dict(id="hr-08", dept="HR", source="payroll_faq.md", year=2024,
         text="Salary is credited on the last working day of each month. Payroll discrepancies must be raised with the payroll desk within 30 days of the pay date."),
    dict(id="hr-09", dept="HR", source="payroll_faq.md", year=2024,
         text="Reimbursement claims for travel and internet are processed in the payroll cycle following approval by the reporting manager."),
    dict(id="hr-10", dept="HR", source="hr_policy.pdf", year=2024,
         text="The hybrid work policy requires employees to be present in the office for a minimum of ten days per month, coordinated with their manager."),
    dict(id="hr-11", dept="HR", source="hr_policy.pdf", year=2024,
         text="Employees may request a sabbatical of up to six months after five years of continuous service, subject to business approval."),
    dict(id="hr-12", dept="HR", source="onboarding.md", year=2024,
         text="New joiners complete mandatory security and code-of-conduct training within the first two weeks of their start date."),

    # ---------------- ENGINEERING ----------------
    dict(id="en-01", dept="ENG", source="auth_service.md", year=2024,
         text="The auth module issues JWT access tokens with a 15 minute expiry and refresh tokens valid for 24 hours. Token lifetime is set by AUTH_TOKEN_TTL."),
    dict(id="en-02", dept="ENG", source="auth_service.md", year=2024,
         text="If clients report tokens expiring early, check for clock skew between the auth pods and the API gateway; skew above 60 seconds invalidates tokens prematurely."),
    dict(id="en-03", dept="ENG", source="k8s_runbook.md", year=2024,
         text="A container exits with OOMKilled when it exceeds its memory limit. Kubernetes terminates the process with exit code 137 and restarts the pod."),
    dict(id="en-04", dept="ENG", source="k8s_runbook.md", year=2024,
         text="To remediate OOMKilled pods, raise resources.limits.memory in the deployment manifest, or profile the service for memory leaks before scaling."),
    dict(id="en-05", dept="ENG", source="contributing.md", year=2024,
         text="Open a pull request against the develop branch. Every PR needs one approving review, a green CI run and a linked ticket before it can be merged."),
    dict(id="en-06", dept="ENG", source="contributing.md", year=2024,
         text="Commit messages follow the conventional commits format. Squash merging is enabled so the PR title becomes the commit message on develop."),
    dict(id="en-07", dept="ENG", source="api_guide.md", year=2024,
         text="The public API is rate limited to 1000 requests per minute per API key. Exceeding the limit returns HTTP 429 with a Retry-After header."),
    dict(id="en-08", dept="ENG", source="api_guide.md", year=2024,
         text="API error code E-4021 indicates a malformed pagination cursor. Reissue the request without the cursor to restart from the first page."),
    dict(id="en-09", dept="ENG", source="db_schema.md", year=2024,
         text="The orders table is partitioned by month. Queries that do not filter on order_date will scan every partition and are the usual cause of slow reports."),
    dict(id="en-10", dept="ENG", source="db_schema.md", year=2024,
         text="Schema migrations run through Alembic in the release pipeline. Backward incompatible column drops must be split across two releases."),
    dict(id="en-11", dept="ENG", source="logging_guide.md", year=2024,
         text="All services emit structured JSON logs with a trace_id field. Use the trace_id to follow one request across the gateway, auth and payment services."),
    dict(id="en-12", dept="ENG", source="python_style.md", year=2024,
         text="Python services target version 3.11, are formatted with black and type-checked with mypy in strict mode on the CI pipeline."),

    # ---------------- OPS ----------------
    dict(id="op-01", dept="OPS", source="incident_playbook.md", year=2024,
         text="Severity 1 incidents require an incident commander within five minutes and a status page update within fifteen minutes of declaration."),
    dict(id="op-02", dept="OPS", source="incident_playbook.md", year=2024,
         text="When the payment container crashes, first check the last deployment, then the pod events, then the upstream provider status page."),
    dict(id="op-03", dept="OPS", source="ci_pipeline.md", year=2024,
         text="A pipeline failure at the build stage is usually a dependency resolution error. Clear the runner cache and rerun before investigating further."),
    dict(id="op-04", dept="OPS", source="ci_pipeline.md", year=2024,
         text="Deployment pipelines block if the staging smoke test suite fails. The smoke tests run against the staging gateway for ninety seconds after rollout."),
    dict(id="op-05", dept="OPS", source="monitoring.md", year=2024,
         text="Latency dashboards are published per region. The Mumbai and US-East deployments are monitored on separate boards with independent alert thresholds."),
    dict(id="op-06", dept="OPS", source="monitoring.md", year=2024,
         text="The Mumbai region reported a p99 latency spike of 2.4 seconds on 14 March, traced to an undersized connection pool on the read replica."),
    dict(id="op-07", dept="OPS", source="monitoring.md", year=2024,
         text="The US-East region remained within its 400 millisecond p99 budget through March and raised no latency alerts during that window."),
    dict(id="op-08", dept="OPS", source="oncall.md", year=2024,
         text="On-call rotations run Monday to Monday. The secondary on-call engineer is paged only if the primary does not acknowledge within ten minutes."),
    dict(id="op-09", dept="OPS", source="oncall.md", year=2024,
         text="Every production incident requires a blameless postmortem published within five working days of resolution."),
    dict(id="op-10", dept="OPS", source="network_manual.pdf", year=2024,
         text="Switch association data is collected only for reporting access points. Rogue units are detected by signal only, so no switch association data exists for them."),
    dict(id="op-11", dept="OPS", source="network_manual.pdf", year=2024,
         text="Network error code NET-503 means the access point lost its uplink to the distribution switch and fell back to mesh mode."),
    dict(id="op-12", dept="OPS", source="backup_policy.md", year=2024,
         text="Production databases are snapshotted every six hours and retained for 30 days. Restore drills are performed quarterly."),
]

DOCS  = [d["text"] for d in CORPUS]
BY_ID = {d["id"]: d for d in CORPUS}

show("Corpus summary", [
    f"passages : {len(CORPUS)}",
    f"depts    : {sorted(set(d['dept'] for d in CORPUS))}",
    f"sources  : {len(set(d['source'] for d in CORPUS))}",
    f"example  : {CORPUS[0]['id']} -> {CORPUS[0]['text'][:60]}...",
])

Corpus summary
passages : 36
depts    : ['ENG', 'HR', 'OPS']
sources  : 17
example  : hr-01 -> Full-time employees accrue 24 days of paid annual leave per ...



In [4]:
QUESTIONS = [
    "How many days of leave do I have left?",   # easy, HR
    "OOMKilled",                                 # jargon fragment, ENG
    "Does it apply to contractors?",             # needs conversation history
    "Pipeline failed.",                          # too vague for keywords
    "Did the latency spike happen in both the Mumbai and US-East deployments?",
    "Why can't I pull switch association data for the rogue units on Floor 3?",
]
for q in QUESTIONS:
    print(" *", q)

 * How many days of leave do I have left?
 * OOMKilled
 * Does it apply to contractors?
 * Pipeline failed.
 * Did the latency spike happen in both the Mumbai and US-East deployments?
 * Why can't I pull switch association data for the rogue units on Floor 3?


### Initialize the Sentence Transformer Model

We'll use the `sentence-transformers` library to load a pre-trained model. This model will convert our text documents and user queries into numerical vectors (embeddings), allowing us to measure their semantic similarity.

In [5]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')

print("Sentence Transformer Model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer Model loaded successfully.
